# Lab 5: Propositional Logic / Пропозицийн логик
## Summary and how to use this notebook

This lab builds a small reasoning engine in Python. It represents sentences as
tuple trees, evaluates their truth in a model, converts sentences into CNF, and
checks logical consequences using **truth tables, resolution, and DPLL**. The
algorithms are then applied to Wumpus World and random 3-SAT. Both bonus exercises
are included below.

The source is the eight-page *Lab5_Logic_Manual.pdf* supplied by the user. The
manual sets a 160-minute lab, references AIMA chapter 7, and expects a Jupyter
notebook. Its instructions are described here as course requirements. This
notebook is a worked learning solution with explanations and executable checks.

| Coursework | Points |
|---|---:|
| Steps 1-3: expressions, semantics, truth tables | 8 |
| Step 4: CNF conversion | 6 |
| Steps 5-6: resolution and DPLL | 10 |
| Steps 7-8: Wumpus, comparison, written answers | 6 |
| Bonus: knights and liars | 6 |
| Bonus: disable DPLL heuristics and compare calls | 4 |

Open this file in Jupyter, select **Python (Lab 1)** in this workspace, and choose
**Run All**. Any Python 3 kernel can run the notebook. It is self-contained and uses
only the Python standard library; all implementation cells are included. Saved
outputs show the actual completed run. The matching module is `lab5_logic.py`.

### Essential vocabulary

- A **symbol** (атом) is a proposition such as `P`: a statement that is True or False.
- A **model** (загвар) assigns a Boolean value to every symbol.
- `KB` is the **knowledge base** (мэдлэгийн сан): the premises assumed together.
- `KB ⊨ α` means every model satisfying KB also satisfies the query α.
- **Valid** means true in every model. **Satisfiable** means true in at least one.
- A **literal** is a symbol or its negation: `P` or `~P`.
- A **clause** is an OR of literals. CNF is an AND of clauses.

The empty clause `frozenset()` is False (a contradiction). The empty set of
clauses `set()` is True (there are no constraints). These must be distinguished.

The key identity is **KB ⊨ α iff KB ∧ ¬α is unsatisfiable**. Truth tables search
for a countermodel directly; resolution tries to derive a contradiction; DPLL
tries to find a model for the premises plus the negated query.

In [21]:
"""Lab 5: propositional logic, CNF, resolution and DPLL.

Standard-library-only implementation. Notebook: Lab5_Logic.ipynb.
Models are dictionaries; clauses are frozensets of strings such as P and ~P.
"""

from itertools import combinations, product
import random

## Step 1: Represent sentences as immutable trees

`Sym('P')` produces `('sym', 'P')`. Unary negation stores one child; binary
connectives store two. For example, `(P ∧ Q) ⇒ R` is
`('imp', ('and', ('sym','P'), ('sym','Q')), ('sym','R'))`.

`symbols` recursively unions the names found in the subtrees. `to_str` prints
parentheses around each connective so its grouping is unambiguous. Constructors
make the trees easier to write. `conjoin` joins several premises using `And`.

In [24]:
def Sym(n):
    if not isinstance(n, str) or not n or n.startswith("~"):
        raise ValueError("A symbol needs a nonempty name without a leading ~")
    return ("sym", n)


def Not(a): return ("not", a)
def And(a, b): return ("and", a, b)
def Or(a, b): return ("or", a, b)
def Imp(a, b): return ("imp", a, b)
def Iff(a, b): return ("iff", a, b)


def symbols(s):
    if s[0] == "sym":
        return {s[1]}
    if s[0] == "not":
        return symbols(s[1])
    return symbols(s[1]) | symbols(s[2])


def to_str(s):
    op = s[0]
    if op == "sym":
        return s[1]
    if op == "not":
        return f"(~{to_str(s[1])})"
    signs = {"and": "&", "or": "|", "imp": "=>", "iff": "<=>"}
    return f"({to_str(s[1])} {signs[op]} {to_str(s[2])})"


def conjoin(sentences):
    """Combine a nonempty sequence of sentences into one binary And tree."""
    it = iter(sentences)
    try:
        result = next(it)
    except StopIteration as exc:
        raise ValueError("conjoin requires at least one sentence") from exc
    for sentence in it:
        result = And(result, sentence)
    return result

In [23]:
P, Q, R = map(Sym, ('P', 'Q', 'R'))
s1 = Imp(And(P, Q), R)
s2 = Iff(P, Q)
s3 = Or(Not(P), Q)
assert symbols(s1) == {'P', 'Q', 'R'}
assert to_str(s1) == '((P & Q) => R)'
for s in (s1, s2, s3):
    print(to_str(s), 'symbols =', sorted(symbols(s)))

((P & Q) => R) symbols = ['P', 'Q', 'R']
(P <=> Q) symbols = ['P', 'Q']
((~P) | Q) symbols = ['P', 'Q']


## Step 2: Evaluate semantics recursively

`pl_true` evaluates each subtree in the same model. Implication is false only
when its premise is true and conclusion is false; equivalence is true when
both sides have the same truth value.

| P | Q | P ∧ Q | P ∨ Q | P ⇒ Q | P ⇔ Q |
|---|---|---|---|---|---|
| False | False | False | False | True | True |
| False | True | False | True | True | False |
| True | False | False | True | False | False |
| True | True | True | True | True | True |

The function expects a complete model. A missing symbol raises `KeyError`, so a
missing assignment cannot silently be mistaken for False.

In [4]:
def pl_true(s, model):
    op = s[0]
    if op == "sym":
        return model[s[1]]
    if op == "not":
        return not pl_true(s[1], model)
    a, b = pl_true(s[1], model), pl_true(s[2], model)
    if op == "and": return a and b
    if op == "or": return a or b
    if op == "imp": return (not a) or b
    if op == "iff": return a == b
    raise ValueError(f"Unknown operator: {op}")

In [5]:
m1 = {'P': True, 'Q': False, 'R': True}
assert pl_true(s1, m1) is True
assert pl_true(s2, m1) is False
assert pl_true(s3, m1) is False
for s in (s1, s2, s3):
    print(to_str(s), '=', pl_true(s, m1))

((P & Q) => R) = True
(P <=> Q) = False
((~P) | Q) = False


## Step 3: Truth tables and entailment

With n symbols there are 2ⁿ complete assignments. `all_models` yields them one
at a time rather than storing them all. To check entailment, evaluate the KB
and the query for each model. If KB is True and the query is False, that model
disproves entailment and the search stops. Models in which KB is False still
count as examined but impose no constraint on the query.

`tt_entails` returns `(answer, examined_models)`. For a true entailment with n
distinct symbols in the KB and query, it examines all 2ⁿ models. A false
entailment may finish earlier. Symbols are sorted, and False precedes True.

Manual checks: Modus Ponens and Modus Tollens hold; `P ∨ Q` does not entail P
because P=False, Q=True is a countermodel. `P ∨ ¬P` is valid; `P ∧ ¬P` has no
satisfying model. De Morgan's law holds row by row.

In [6]:
def all_models(syms):
    """Sorted names; False precedes True, for reproducible counters."""
    names = sorted(set(syms))
    for values in product((False, True), repeat=len(names)):
        yield dict(zip(names, values))


def tt_entails(kb, alpha, max_models=None):
    """Return (entailed, models examined); None means the limit was exceeded."""
    if isinstance(kb, list):
        kb = conjoin(kb)
    count = 0
    for model in all_models(symbols(kb) | symbols(alpha)):
        count += 1
        if max_models is not None and count > max_models:
            return None, count - 1
        if pl_true(kb, model) and not pl_true(alpha, model):
            return False, count
    return True, count


def is_valid(s):
    return all(pl_true(s, m) for m in all_models(symbols(s)))


def is_satisfiable(s):
    return any(pl_true(s, m) for m in all_models(symbols(s)))


def print_truth_table(s):
    names = sorted(symbols(s))
    print(" | ".join(names + [to_str(s)]))
    for model in all_models(names):
        print(" | ".join(str(int(model[n])) for n in names)
              + " | " + str(int(pl_true(s, model))))

In [7]:
checks = [
    ('Modus Ponens', And(P, Imp(P, Q)), Q, True),
    ('Disjunction does not entail P', Or(P, Q), P, False),
    ('Modus Tollens', And(Not(Q), Imp(P, Q)), Not(P), True),
]
for label, premises, query, expected in checks:
    answer, count = tt_entails(premises, query)
    assert answer is expected
    print(label, ':', answer, '; models =', count)
assert is_valid(Or(P, Not(P)))
assert not is_satisfiable(And(P, Not(P)))
print('P | ~P is valid:', is_valid(Or(P, Not(P))))
print('P & ~P is satisfiable:', is_satisfiable(And(P, Not(P))))
left, right = Not(And(P, Q)), Or(Not(P), Not(Q))
print('\nDe Morgan: left side')
print_truth_table(left)
print('\nDe Morgan: right side')
print_truth_table(right)
assert all(pl_true(left, m) == pl_true(right, m) for m in all_models({'P', 'Q'}))

Modus Ponens : True ; models = 4
Disjunction does not entail P : False ; models = 2
Modus Tollens : True ; models = 4
P | ~P is valid: True
P & ~P is satisfiable: False

De Morgan: left side
P | Q | (~(P & Q))
0 | 0 | 1
0 | 1 | 1
1 | 0 | 1
1 | 1 | 0

De Morgan: right side
P | Q | ((~P) | (~Q))
0 | 0 | 1
0 | 1 | 1
1 | 0 | 1
1 | 1 | 0


## Step 4: Convert to conjunctive normal form

Apply these transformations in order, without modifying the original tuples:

1. Replace `A ⇔ B` with `(A ⇒ B) ∧ (B ⇒ A)`.
2. Replace `A ⇒ B` with `¬A ∨ B`.
3. Push negations down using De Morgan's laws and remove double negation.
4. Distribute OR over AND: `A ∨ (B ∧ C)` becomes `(A ∨ B) ∧ (A ∨ C)`.

Example:

`B ⇔ (A ∨ C)`

`≡ (¬B ∨ A ∨ C) ∧ (¬(A ∨ C) ∨ B)`

`≡ (¬B ∨ A ∨ C) ∧ (¬A ∨ B) ∧ (¬C ∨ B)`.

The three clauses are `{~B, A, C}`, `{~A, B}`, `{~C, B}`.
`to_clauses` takes an already-CNF expression; `clausify` combines conversion
and extraction. Tautological clauses such as `{P, ~P}` are omitted because
they are true under every assignment. Distribution can increase expression
size greatly; this lab uses direct logical equivalences.

In [8]:
def eliminate_iff(s):
    op = s[0]
    if op == "sym": return s
    if op == "not": return Not(eliminate_iff(s[1]))
    a, b = eliminate_iff(s[1]), eliminate_iff(s[2])
    if op == "iff": return And(Imp(a, b), Imp(b, a))
    return (op, a, b)


def eliminate_imp(s):
    op = s[0]
    if op == "sym": return s
    if op == "not": return Not(eliminate_imp(s[1]))
    a, b = eliminate_imp(s[1]), eliminate_imp(s[2])
    if op == "imp": return Or(Not(a), b)
    if op == "iff":
        raise ValueError("Run eliminate_iff before eliminate_imp")
    return (op, a, b)


def push_not(s):
    op = s[0]
    if op == "sym": return s
    if op == "not":
        a = s[1]
        if a[0] == "sym": return s
        if a[0] == "not": return push_not(a[1])
        if a[0] == "and": return Or(push_not(Not(a[1])), push_not(Not(a[2])))
        if a[0] == "or": return And(push_not(Not(a[1])), push_not(Not(a[2])))
        raise ValueError("Remove implications before push_not")
    if op not in ("and", "or"):
        raise ValueError("Remove implications before push_not")
    return (op, push_not(s[1]), push_not(s[2]))


def distribute(s):
    if s[0] in ("sym", "not"): return s
    a, b = distribute(s[1]), distribute(s[2])
    if s[0] == "and": return And(a, b)
    if s[0] != "or": raise ValueError("distribute expects and/or and literals")
    if a[0] == "and":
        return And(distribute(Or(a[1], b)), distribute(Or(a[2], b)))
    if b[0] == "and":
        return And(distribute(Or(a, b[1])), distribute(Or(a, b[2])))
    return Or(a, b)


def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))


def opposite(literal):
    return literal[1:] if literal.startswith("~") else "~" + literal


def literal_symbol(literal):
    return literal[1:] if literal.startswith("~") else literal


def is_tautology_clause(clause):
    return any(opposite(lit) in clause for lit in clause)


def to_clauses(s):
    """Input is a CNF expression; remove duplicate and tautological clauses."""
    if s[0] == "and":
        return to_clauses(s[1]) | to_clauses(s[2])

    def literals(expr):
        if expr[0] == "sym": return {expr[1]}
        if expr[0] == "not" and expr[1][0] == "sym": return {"~" + expr[1][1]}
        if expr[0] == "or": return literals(expr[1]) | literals(expr[2])
        raise ValueError("to_clauses expects CNF; call to_cnf first")

    clause = frozenset(literals(s))
    return set() if is_tautology_clause(clause) else {clause}


def clausify(s):
    return to_clauses(to_cnf(s))


def clause_key(clause):
    return len(clause), tuple(sorted(clause))


def format_clauses(clauses):
    return " & ".join("{" + ", ".join(sorted(c)) + "}"
                      for c in sorted(clauses, key=clause_key)) or "TRUE (no clauses)"

In [9]:
A, B, C = map(Sym, ('A', 'B', 'C'))
example = Iff(B, Or(A, C))
expected = {frozenset(('~B', 'A', 'C')), frozenset(('~A', 'B')), frozenset(('~C', 'B'))}
assert clausify(example) == expected
print('CNF:', to_str(to_cnf(example)))
print('Clauses:', format_clauses(clausify(example)))
examples = [example, Imp(And(A, B), C), Not(Or(A, And(B, C))), Or(A, And(B, C))]
for sentence in examples:
    converted = to_cnf(sentence)
    assert all(pl_true(sentence, m) == pl_true(converted, m)
               for m in all_models(symbols(sentence)))
    print('Equivalent in every model:', to_str(sentence))
assert clausify(Or(A, Not(A))) == set()

CNF: (((~B) | (A | C)) & (((~A) | B) & ((~C) | B)))
Clauses: {B, ~A} & {B, ~C} & {A, C, ~B}
Equivalent in every model: (B <=> (A | C))
Equivalent in every model: ((A & B) => C)
Equivalent in every model: (~(A | (B & C)))
Equivalent in every model: (A | (B & C))


## Step 5: Resolution by refutation

The resolution rule combines `{A, B}` and `{~A, C}` into `{B, C}`. Exactly one
complementary pair is removed for each candidate. Removing several pairs at
once is unsound. Candidates containing a literal and its negation are discarded.

To prove `KB ⊨ α`, add the clauses of `¬α`, resolve all clause pairs, and keep
adding new clauses. Deriving the empty clause proves entailment. If an entire
round adds nothing, the clause set is satisfiable and entailment fails.

Worked example: KB is `(A ∨ B) ∧ (¬A ∨ C) ∧ (¬B ∨ C)`, query C.
Add `{~C}`. Resolving with `{~A, C}` gives `{~A}`; resolving with `{~B, C}`
gives `{~B}`. Resolve `{A, B}` with `{~A}` to obtain `{B}`; resolve that
with `{~B}` to obtain `{}`. Therefore KB entails C. It does not entail A:
`A=False, B=True, C=True` satisfies KB and falsifies A.

`pl_resolution` returns an entailment Boolean. The optional `stats` dictionary
records `resolvents`. All retained resolvents are counted, including those
already present; rejected tautologies do not count. Every search resets its counter.

In [10]:
def resolve(ci, cj):
    """Resolve one complementary pair at a time, discarding tautologies."""
    result = []
    for lit in sorted(ci):
        neg = opposite(lit)
        if neg in cj:
            resolvent = frozenset((set(ci) - {lit}) | (set(cj) - {neg}))
            if not is_tautology_clause(resolvent):
                result.append(resolvent)
    return result


def resolution_unsat(clauses, stats=None, max_resolvents=None):
    """True=UNSAT, False=SAT, None=limit; counts all retained resolvents.

    Full pair enumeration each round follows the manual. Already-known
    resolvents are counted too; tautological candidates are not counted.
    """
    stats = {} if stats is None else stats
    stats["resolvents"] = 0
    clauses = {frozenset(c) for c in clauses if not is_tautology_clause(c)}
    if frozenset() in clauses:
        return True
    while True:
        new = set()
        for ci, cj in combinations(sorted(clauses, key=clause_key), 2):
            for resolvent in resolve(ci, cj):
                stats["resolvents"] += 1
                if max_resolvents is not None and stats["resolvents"] > max_resolvents:
                    return None
                if not resolvent:
                    return True
                new.add(resolvent)
        if new <= clauses:
            return False
        clauses |= new


def pl_resolution(kb_clauses, alpha, stats=None, max_resolvents=None):
    return resolution_unsat(set(kb_clauses) | clausify(Not(alpha)),
                            stats, max_resolvents)

In [11]:
assert resolve(frozenset(('A', 'B')), frozenset(('~A', 'C'))) == [frozenset(('B', 'C'))]
assert resolve(frozenset(('C',)), frozenset(('~C',))) == [frozenset()]
kb5 = conjoin([Or(A, B), Or(Not(A), C), Or(Not(B), C)])
for query, expected in ((C, True), (A, False)):
    stats = {}
    result = pl_resolution(clausify(kb5), query, stats)
    tt, count = tt_entails(kb5, query)
    assert result == tt == expected
    print('Query:', to_str(query), '; entailed =', result,
          '; resolvents =', stats['resolvents'], '; TT models =', count)

Query: C ; entailed = True ; resolvents = 17 ; TT models = 8
Query: A ; entailed = False ; resolvents = 17 ; TT models = 4


## Step 6: DPLL satisfiability search

DPLL searches partial assignments. A clause can be satisfied, contradicted,
or still undecided. A contradicted clause rejects the branch immediately;
if every clause is satisfied, the current assignment is a solution.

The manual's order is followed: first try a **pure literal**, then a **unit
clause**, then branch True/False on the first remaining sorted symbol.
A pure symbol appears in only one polarity among unsatisfied clauses, so that
polarity can be chosen without losing satisfiability. A unit clause has only
one unassigned literal and all other literals false, so its value is forced.

Example: `{A}`, `{~A,B}`, `{~B,C}` force A=True, then B=True, then C=True.
With `{~C,D,E}` and `{~D,~E}`, at least one of D,E must be true and they
cannot both be true: exactly one is true.

`dpll_satisfiable` returns a satisfying model or **False**. It completes any
irrelevant remaining symbols with False. Test success using `model is not False`,
because `{}` is the valid model for an empty clause set. Its optional `stats`
dictionary stores `calls`. For entailment, a False result on `KB ∧ ¬α` means
the query is entailed; a returned model is a counterexample.

In [12]:
def clause_value(clause, model):
    """True=satisfied, False=contradicted, None=still undecided."""
    unknown = False
    for lit in clause:
        name = literal_symbol(lit)
        if name not in model:
            unknown = True
        elif model[name] == (not lit.startswith("~")):
            return True
    return None if unknown else False


def find_pure_symbol(symbols, clauses, model):
    """Consider only clauses not already satisfied by the model."""
    active = [c for c in clauses if clause_value(c, model) is not True]
    present = set().union(*active) if active else set()
    for name in sorted(symbols):
        if name in model: continue
        positive, negative = name in present, "~" + name in present
        if positive != negative:
            return name, positive
    return None


def find_unit_clause(clauses, model):
    for clause in sorted(clauses, key=clause_key):
        if clause_value(clause, model) is True: continue
        remaining = [lit for lit in clause if literal_symbol(lit) not in model]
        if len(remaining) == 1:
            lit = remaining[0]
            return literal_symbol(lit), not lit.startswith("~")
    return None


def dpll(clauses, symbols, model, *, stats=None, use_unit=True, use_pure=True):
    stats = {"calls": 0} if stats is None else stats
    stats["calls"] = stats.get("calls", 0) + 1
    active = []
    for clause in clauses:
        value = clause_value(clause, model)
        if value is False: return False
        if value is None: active.append(clause)
    if not active: return dict(model)

    def assign(name, value):
        return dpll(clauses, set(symbols) - {name}, {**model, name: value},
                    stats=stats, use_unit=use_unit, use_pure=use_pure)

    forced = find_pure_symbol(symbols, active, model) if use_pure else None
    if forced is None and use_unit:
        forced = find_unit_clause(active, model)
    if forced is not None:
        return assign(*forced)
    name = sorted(symbols)[0]
    answer = assign(name, True)
    # {} is a successful model for the empty conjunction; never use `if answer`.
    if answer is not False: return answer
    return assign(name, False)


def dpll_satisfiable(clauses, stats=None, *, use_unit=True, use_pure=True):
    """Return a complete satisfying model or False; optional stats holds calls."""
    stats = {} if stats is None else stats
    stats["calls"] = 0
    clauses = tuple(sorted({frozenset(c) for c in clauses}, key=clause_key))
    names = {literal_symbol(lit) for c in clauses for lit in c}
    answer = dpll(clauses, names, {}, stats=stats,
                  use_unit=use_unit, use_pure=use_pure)
    if answer is False: return False
    # Unassigned names are irrelevant once every clause already has a true literal.
    return {name: answer.get(name, False) for name in sorted(names)}

In [13]:
sat6 = {frozenset(c) for c in [('A',), ('~A','B'), ('~B','C'), ('~C','D','E'), ('~D','~E')]}
stats = {}
model = dpll_satisfiable(sat6, stats)
assert model is not False
assert model['A'] and model['B'] and model['C'] and model['D'] != model['E']
assert all(clause_value(c, model) is True for c in sat6)
print('Satisfying model:', model, '; calls =', stats['calls'])
unsat6 = {frozenset(c) for c in [('A','B'), ('~A','B'), ('A','~B'), ('~A','~B')]}
stats = {}
assert dpll_satisfiable(unsat6, stats) is False
print('Four-clause example: UNSAT; calls =', stats['calls'])
assert dpll_satisfiable(clausify(And(kb5, Not(C)))) is False
print('KB from Step 5 with ~C: UNSAT, so KB entails C')

Satisfying model: {'A': True, 'B': True, 'C': True, 'D': True, 'E': False} ; calls = 6
Four-clause example: UNSAT; calls = 5
KB from Step 5 with ~C: UNSAT, so KB entails C


## Step 7: Wumpus World knowledge base

`Pxy` means there is a pit in cell [x,y]; `Bxy` means a breeze is sensed there.
The five premises are:

1. `¬P11`
2. `B11 ⇔ (P12 ∨ P21)`
3. `B21 ⇔ (P11 ∨ P22 ∨ P31)`
4. `¬B11`
5. `B21`

Since B11 is false, P12 and P21 must both be false. Since B21 is true and
P11 is false, at least one of P22 and P31 is true. We cannot identify which
one from this KB. The three satisfying possibilities for `(P22,P31)` are
`(False,True)`, `(True,False)`, `(True,True)`. All other values are fixed:
B11=False, B21=True, P11=P12=P21=False.

| Query α | KB ⊨ α | Explanation |
|---|---|---|
| ¬P12 | True | No breeze at [1,1] excludes a pit at [1,2]. |
| ¬P21 | True | The same rule excludes a pit at [2,1]. |
| ¬P22 | False | A pit at [2,2] is compatible with KB. |
| ¬P31 | False | A pit at [3,1] is compatible with KB. |
| P22 ∨ P31 | True | The breeze at [2,1] requires one of these pits. |

False entailment means the KB permits a counterexample. It does not by itself
mean the opposite query is entailed. Here neither a pit nor its absence is
entailed individually for [2,2] or [3,1].

In [14]:
def wumpus_problem():
    P11, P12, P21, P22, P31, B11, B21 = map(
        Sym, ("P11", "P12", "P21", "P22", "P31", "B11", "B21"))
    kb = conjoin([
        Not(P11), Iff(B11, Or(P12, P21)),
        Iff(B21, Or(Or(P11, P22), P31)), Not(B11), B21,
    ])
    return kb, [Not(P12), Not(P21), Not(P22), Not(P31), Or(P22, P31)]


def compare_entailment(kb, alpha):
    tt, models = tt_entails(kb, alpha)
    rstats, dstats = {}, {}
    resolution = pl_resolution(clausify(kb), alpha, rstats)
    countermodel = dpll_satisfiable(clausify(And(kb, Not(alpha))), dstats)
    dpll_answer = countermodel is False
    assert tt == resolution == dpll_answer
    return {"query": to_str(alpha), "entailed": tt, "tt_models": models,
            "resolution_resolvents": rstats["resolvents"],
            "dpll_calls": dstats["calls"], "countermodel": countermodel}

In [15]:
wumpus_kb, queries = wumpus_problem()
wumpus_models = [m for m in all_models(symbols(wumpus_kb)) if pl_true(wumpus_kb, m)]
assert len(wumpus_models) == 3
print('Satisfying KB models:', len(wumpus_models))
for m in wumpus_models:
    print(m)
wumpus_results = [compare_entailment(wumpus_kb, q) for q in queries]
assert [r['entailed'] for r in wumpus_results] == [True, True, False, False, True]
print('\n| Query | TT | Resolution | DPLL |')
print('|---|---|---|---|')
for row in wumpus_results:
    a = row['entailed']
    print(f"| {row['query'].replace('|', '∨')} | {a} | {a} | {a} |")
for row in wumpus_results:
    if row['countermodel'] is not False:
        print('Countermodel for', row['query'], ':', row['countermodel'])

Satisfying KB models: 3
{'B11': False, 'B21': True, 'P11': False, 'P12': False, 'P21': False, 'P22': False, 'P31': True}
{'B11': False, 'B21': True, 'P11': False, 'P12': False, 'P21': False, 'P22': True, 'P31': False}
{'B11': False, 'B21': True, 'P11': False, 'P12': False, 'P21': False, 'P22': True, 'P31': True}

| Query | TT | Resolution | DPLL |
|---|---|---|---|
| (~P12) | True | True | True |
| (~P21) | True | True | True |
| (~P22) | False | False | False |
| (~P31) | False | False | False |
| (P22 ∨ P31) | True | True | True |
Countermodel for (~P22) : {'B11': False, 'B21': True, 'P11': False, 'P12': False, 'P21': False, 'P22': True, 'P31': False}
Countermodel for (~P31) : {'B11': False, 'B21': True, 'P11': False, 'P12': False, 'P21': False, 'P22': True, 'P31': True}


## Step 8: Measured algorithm comparison

The first three rows check **entailment**. The 3-SAT rows check **satisfiability**
of the generated formula itself. Resolution's raw result means UNSAT, so it is
inverted for SAT rows. DPLL's raw model/False result is also translated to the
appropriate question before comparison.

Each random problem has n=10,15,20 and m=4n clauses, each with three distinct
variables. We use seed=1. A local `random.Random(1)` has the same seeded sampling
behavior as `random.seed(1)` but does not alter unrelated random code. For signs,
`choice((True, False))` selects positive then negative; duplicate clauses are
resampled to obtain exactly m distinct clauses. Names X01,... sort numerically.
The sign-choice convention and duplicate handling are recorded because the
manual does not fully specify them; another implementation can produce different counts.

Limits: truth-table search examines at most **200,000** models. Resolution stops
when its generated-resolvent count exceeds **100,000**, at 100,001. Such a row is
shown as **хэт удаан / limit** and is inconclusive; a cutoff is not an UNSAT result.
Counters reset before each problem. Truth-table counts, resolvent counts, and
DPLL calls measure different kinds of work, so their sizes are not timings.

In [16]:
def random_3sat(n, m, seed=1):
    """Exactly m distinct clauses; local Random(seed) avoids global side effects.

    Names X01,...,X20 sort numerically. Sample three different variable indices;
    choose a positive/negative sign with choice((True, False)). Resample duplicates.
    """
    if n < 3 or not 0 <= m <= 8 * (n * (n - 1) * (n - 2) // 6):
        raise ValueError("Invalid n or m for distinct three-literal clauses")
    rng = random.Random(seed)
    clauses = set()
    while len(clauses) < m:
        indices = rng.sample(range(1, n + 1), 3)
        clauses.add(frozenset(("" if rng.choice((True, False)) else "~")
                              + f"X{i:02d}" for i in indices))
    return clauses


def tt_sat_clauses(clauses, max_models=None):
    """SAT by exhaustive models: return (True/False/None, count)."""
    clauses = tuple(sorted(clauses, key=clause_key))
    names = {literal_symbol(lit) for c in clauses for lit in c}
    count = 0
    for model in all_models(names):
        count += 1
        if max_models is not None and count > max_models:
            return None, count - 1
        if all(clause_value(c, model) is True for c in clauses):
            return True, count
    return False, count


def benchmark():
    rows = []
    kb, queries = wumpus_problem()
    for index in (0, 2, 4):
        result = compare_entailment(kb, queries[index])
        rows.append({"problem": "Wumpus: " + result["query"],
                     "task": "entailment", "tt_result": result["entailed"],
                     "tt_count": result["tt_models"],
                     "resolution_result": result["entailed"],
                     "resolution_count": result["resolution_resolvents"],
                     "dpll_result": result["entailed"],
                     "dpll_count": result["dpll_calls"]})
    for n in (10, 15, 20):
        clauses = random_3sat(n, 4*n, seed=1)
        tt, models = tt_sat_clauses(clauses, max_models=200_000)
        rstats, dstats = {}, {}
        unsat = resolution_unsat(clauses, rstats, max_resolvents=100_000)
        model = dpll_satisfiable(clauses, dstats)
        sat = model is not False
        resolution_sat = None if unsat is None else not unsat
        assert tt is None or tt == sat
        assert resolution_sat is None or resolution_sat == sat
        if sat: assert all(clause_value(c, model) is True for c in clauses)
        rows.append({"problem": f"3-SAT n={n}, m={4*n}, seed=1", "task": "SAT",
                     "tt_result": tt, "tt_count": models,
                     "resolution_result": resolution_sat,
                     "resolution_count": rstats["resolvents"],
                     "dpll_result": sat, "dpll_count": dstats["calls"]})
    return rows

In [17]:
benchmark_rows = benchmark()
def measured_cell(row, method):
    result, count = row[method + '_result'], row[method + '_count']
    if result is None:
        return f'хэт удаан / limit ({count:,})'
    label = str(result) if row['task'] == 'entailment' else ('SAT' if result else 'UNSAT')
    return f'{label} ({count:,})'
print('| Problem | Truth table: models | Resolution: resolvents | DPLL: calls |')
print('|---|---|---|---|')
for row in benchmark_rows:
    name = row['problem'].replace('|', '∨')
    print('| ' + ' | '.join([name] + [measured_cell(row, m) for m in ('tt', 'resolution', 'dpll')]) + ' |')

| Problem | Truth table: models | Resolution: resolvents | DPLL: calls |
|---|---|---|---|
| Wumpus: (~P12) | True (128) | True (6) | True (7) |
| Wumpus: (~P22) | False (35) | False (127) | False (7) |
| Wumpus: (P22 ∨ P31) | True (128) | True (159) | True (8) |
| 3-SAT n=10, m=40, seed=1 | SAT (670) | хэт удаан / limit (100,001) | SAT (11) |
| 3-SAT n=15, m=60, seed=1 | SAT (2,112) | хэт удаан / limit (100,001) | SAT (44) |
| 3-SAT n=20, m=80, seed=1 | хэт удаан / limit (200,000) | хэт удаан / limit (100,001) | SAT (60) |


### Answers to the three comparison questions

**1. Which method used the fewest counted operations on the small Wumpus KB?**
Resolution used 6 resolvents for ¬P12 versus DPLL's 7 calls and TT's 128 models.
DPLL was lowest on the other two selected queries: 7 calls for ¬P22 and 8 for
P22 ∨ P31. Across those three rows, DPLL totals 22 calls, resolution 292
resolvents, and TT 291 models. These totals describe the chosen counters;
they do not establish an equivalent amount of work or elapsed time per operation.

**2. Why does truth-table work grow sharply as n increases, and why does DPLL
often use fewer calls?** There are 2ⁿ assignments: 1,024 for n=10, 32,768 for
n=15, and 1,048,576 for n=20. Five additional variables multiply the search
space by 32. SAT search can stop at its first solution, so a measured SAT count
need not grow monotonically; proving UNSAT exhaustively requires all models.
DPLL rejects a branch as soon as a clause is false and uses forced or safe
assignments to shrink the search. It still has exponential worst-case behavior.
In this run DPLL uses 11, 44, and 60 calls on the three random problems.

**3. Why does resolution generate many clauses on satisfiable problems?**
There is no empty-clause proof to finish the search early. This implementation
must reach closure, where every resolvent is already known, to establish SAT.
With n symbols there can be up to 3ⁿ non-tautological clauses, since a symbol
can appear positive, negative, or be absent. Repeated pair enumeration also
regenerates known resolvents. All three random resolution runs reached their
cutoff, even though DPLL found and verified satisfying models.

## Bonus: Knights and liars / Рыцарь ба луйварчин

Let KA, KB, KC mean that A, B, C respectively are knights. A knight's statement
is true and a liar's statement is false, so each person gives `KX ⇔ statement`:

- A says B is a liar: `KA ⇔ ¬KB`.
- B says A and C are the same type: `KB ⇔ (KA ⇔ KC)`.
- C says B is a knight: `KC ⇔ KB`.

Thus KA is the opposite of KB, while KC equals KB. A and C are consequently
always different, so B's statement is false. B must be a liar: KB=False.
Then KA=True and KC=False. The unique solution is **A knight, B liar, C liar**.
The code also implements the second bonus task: DPLL with each heuristic enabled
or disabled. Search outcomes must stay the same, though call counts can change.

In [25]:
def knights_problem():
    KA, KB, KC = map(Sym, ("KA", "KB", "KC"))
    return conjoin([Iff(KA, Not(KB)), Iff(KB, Iff(KA, KC)), Iff(KC, KB)])


def heuristic_comparison():
    """Four DPLL variants on all six comparison problems."""
    kb, queries = wumpus_problem()
    problems = [("Wumpus: " + to_str(queries[i]),
                 clausify(And(kb, Not(queries[i])))) for i in (0, 2, 4)]
    problems += [(f"3-SAT n={n}", random_3sat(n, 4*n, 1)) for n in (10, 15, 20)]
    rows = []
    for name, clauses in problems:
        row = {"problem": name}
        outcomes = []
        for label, unit, pure in (("both", True, True), ("unit_only", True, False),
                                  ("pure_only", False, True), ("neither", False, False)):
            stats = {}
            model = dpll_satisfiable(clauses, stats, use_unit=unit, use_pure=pure)
            outcomes.append(model is not False)
            if model is not False:
                assert all(clause_value(c, model) is True for c in clauses)
            row[label] = stats["calls"]
        assert len(set(outcomes)) == 1
        rows.append(row)
    return rows

In [26]:
knights_kb = knights_problem()
knights_models = [m for m in all_models(symbols(knights_kb)) if pl_true(knights_kb, m)]
assert knights_models == [{'KA': True, 'KB': False, 'KC': False}]
print('Unique model:', knights_models[0])
for query in (Sym('KA'), Not(Sym('KB')), Not(Sym('KC'))):
    answer, count = tt_entails(knights_kb, query)
    assert answer is True and count == 8
    print('Entailed:', to_str(query), '; models examined =', count)

Unique model: {'KA': True, 'KB': False, 'KC': False}
Entailed: KA ; models examined = 8
Entailed: (~KB) ; models examined = 8
Entailed: (~KC) ; models examined = 8


In [27]:
heuristic_rows = heuristic_comparison()
print('| Problem | Both | Unit only | Pure only | Neither |')
print('|---|---:|---:|---:|---:|')
for row in heuristic_rows:
    print('| ' + row['problem'].replace('|', '∨') + ' | '
          + ' | '.join(str(row[k]) for k in ('both', 'unit_only', 'pure_only', 'neither')) + ' |')

| Problem | Both | Unit only | Pure only | Neither |
|---|---:|---:|---:|---:|
| Wumpus: (~P12) | 7 | 4 | 10 | 9 |
| Wumpus: (~P22) | 7 | 7 | 8 | 11 |
| Wumpus: (P22 ∨ P31) | 8 | 6 | 13 | 15 |
| 3-SAT n=10 | 11 | 10 | 11 | 18 |
| 3-SAT n=15 | 44 | 32 | 183 | 555 |
| 3-SAT n=20 | 60 | 78 | 1224 | 1901 |


### Interpreting the bonus comparison

Turning both rules off increases calls on all six selected problems in this run.
The larger SAT examples change from 44 to 555 calls (n=15) and from 60 to 1,901
calls (n=20). Unit propagation is especially helpful here.

Enabling both rules need not give the smallest count in every case. For ¬P12,
unit-only DPLL reaches a contradiction in 4 calls, while the manual's pure-first
order takes 7. Pure assignments can spend calls on variables that unit
propagation would not need before finding a contradiction. For n=20, however,
using both rules improves on unit-only (60 versus 78). The benefit depends on
the formula and on the order in which assignments and branches are chosen.

### What to be able to explain in class

Explain how the tuple tree is evaluated, why a false entailment has a countermodel,
why CNF conversion preserves every model, how the empty clause proves a contradiction,
and how DPLL reduces the search. Describe the two counter limits, the different
units of the three counters, and the measured benefit of the two heuristics.

The separate `test_lab5_logic.py` checks every manual checkpoint and cross-checks
80 small randomly generated CNF formulas against exhaustive truth tables with
all four DPLL variants. It also checks CNF equivalence on 40 random expressions.